In [ ]:
# Preface 分块 + LangSmith/OpenAI Key 配置（换成真实 key；需在导入 langchain 前设置）
#Preface: Chunking
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = "<your-api-key>"
os.environ['OPENAI_API_KEY'] = "<your-api-key>"

In [ ]:
#Part 15: Re-ranking

In [ ]:
# Part 15 重排：索引阶段 —— 加载博客 -> tiktoken 切分(300/50) -> 存入 Chroma 得到 retriever
# 修红：RecursiveCharacterTextSplitter 在 1.x 迁到 langchain_text_splitters
#### INDEXING ####

# Load blog
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)

# Index
from langchain_openai import OpenAIEmbeddings
# from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits,
                                    # embedding=CohereEmbeddings()
                                    embedding=OpenAIEmbeddings())


retriever = vectorstore.as_retriever()

In [ ]:
# RAG-Fusion：生成 4 个相关搜索查询的 Prompt
# 修红：ChatPromptTemplate 在 1.x 迁到 langchain_core.prompts
from langchain_core.prompts import ChatPromptTemplate

# RAG-Fusion
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [ ]:
# 生成查询链：Prompt -> LLM -> 按行切分成多个查询
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_queries = (
    prompt_rag_fusion
    | ChatOpenAI(temperature=0)
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

In [ ]:
# RRF 倒数排名融合：对多路检索结果按排名重新打分排序（k 为平滑常数）
# 修红：dumps/loads 在 1.x 迁到 langchain_core.load
from langchain_core.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """ 倒数排名融合（RRF）：接收多个「已排序文档列表」，
        以及 RRF 公式中可选参数 k """

    # 初始化一个字典，用于保存每个唯一文档的融合分数
    fused_scores = {}

    # 遍历每一个「已排序文档列表」
    for docs in results:
        # 遍历列表中的每个文档，并取其排名（在列表中的位置）
        for rank, doc in enumerate(docs):
            # 将文档转换为字符串作为字典的 key（假设文档可序列化为 JSON）
            doc_str = dumps(doc)
            # 如果该文档还不在 fused_scores 中，则以初始分数 0 加入
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # 取出该文档当前的分数（如果有）
            previous_score = fused_scores[doc_str]
            # 用 RRF 公式更新该文档的分数：1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # 按融合分数降序排序，得到最终重排后的结果
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # 以（文档, 融合分数）元组列表的形式返回重排结果
    return reranked_results

question = "What is task decomposition for LLM agents?"
retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

In [ ]:
# 用 RRF 融合后的 context 组装 RAG 链回答
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(temperature=0)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

In [ ]:
# Cohere Rerank 上下文压缩重排所需的导入
# 修红：langchain.retrievers[.document_compressors] 在 1.x 迁到 langchain_classic
# 注意：CohereRerank 运行需 `pip install cohere` 且设置 COHERE_API_KEY
from langchain_community.llms import Cohere
from langchain_classic.retrievers import  ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CohereRerank

In [ ]:
# 上下文压缩重排：先用向量检索取回 10 条，再用 Cohere 重排挑出最相关的
# 修红：import 迁到 langchain_classic；get_relevant_documents 已移除，改用 invoke
from langchain_classic.retrievers.document_compressors import CohereRerank

retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# Re-rank
compressor = CohereRerank()
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(question)

In [ ]:
#16 - Retrieval (CRAG)

In [ ]:
# 16-18 部分：CRAG / Self-RAG / 长上下文影响 —— 原教程这里只是外部链接说明，无可运行代码
# 16 - Retrieval (CRAG)
# Deep Dive
#
# https://www.youtube.com/watch?v=E2shqsYwxck
#
# Notebooks
#
# https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_crag.ipynb
#
# https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_crag_mistral.ipynb
#
# Generation
# 17 - Retrieval (Self-RAG)
# Notebooks
#
# https://github.com/langchain-ai/langgraph/tree/main/examples/rag
#
# https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_self_rag_mistral_nomic.ipynb
#
# 18 - Impact of long context
# Deep dive
#
# https://www.youtube.com/watch?v=SsHUNfhF32s
#
# Slides
#
# https://docs.google.com/presentation/d/1mJUiPBdtf58NfuSEQ7pVSEQ2Oqmek7F1i4gBwR6JDss/edit#slide=id.g26c0cb8dc66_0_0